In [1]:
import uxarray as ux
import xarray as xr
import numpy as np
import geocat.datafiles as gdf
import geocat.comp as gc


/Users/jkent/miniconda3/envs/uxarray-where/lib/python3.13/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:
data_path = 'netcdf_files/e2p3b09.F2000climo.ne30pg3.ctl002.cam.h0.0005-01.zonal_mpsi_subset.nc'
grid_path = 'netcdf_files/ne30pg3_scrip_170604.nc'
uxds = ux.open_dataset(gdf.get(grid_path), gdf.get(data_path)).drop('lat') #dropping lat to avoid using it instead of the UXarray grid

uxds

/var/folders/dd/_xm_pbpd3flgbvbnt7qhd70snnbpj_/T/ipykernel_24131/1643203269.py:3: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  uxds = ux.open_dataset(gdf.get(grid_path), gdf.get(data_path)).drop('lat') #dropping lat to avoid using it instead of the UXarray grid


<xarray.UxDataset> Size: 6MB
Dimensions:  (time: 1, n_face: 48600, lev: 32, ilev: 33)
Coordinates:
  * time     (time) object 8B 0005-02-01 00:00:00
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * ilev     (ilev) float64 264B 2.255 5.032 10.16 18.56 ... 967.5 985.1 1e+03
Dimensions without coordinates: n_face
Data variables:
    PS       (time, n_face) float32 194kB ...
    V        (time, lev, n_face) float32 6MB ...
    hyam     (lev) float64 256B ...
    hybm     (lev) float64 256B ...
    hyai     (ilev) float64 264B ...
    hybi     (ilev) float64 264B ...

In [3]:
center_coord = [-87.6298, 41.8781]

V_sub = uxds["V"].subset.nearest_neighbor(
    center_coord, k=30, element="face centers"
)
PS_sub = uxds["PS"].subset.nearest_neighbor(
    center_coord, k=30, element="face centers"
)

In [5]:
V_sub

<xarray.UxDataArray 'V' (time: 1, lev: 32, n_face: 30)> Size: 4kB
[960 values with dtype=float32]
Coordinates:
  * time     (time) object 8B 0005-02-01 00:00:00
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
Dimensions without coordinates: n_face
Attributes:
    mdims:         1
    units:         m/s
    long_name:     Meridional wind
    cell_methods:  time: mean

In [4]:
print("plev size:", V_sub.plev.size)
print("plev min/max:", V_sub.plev.min().values, V_sub.plev.max().values)
print("PS min/max:", PS_sub.min().values, PS_sub.max().values)


AttributeError: 'UxDataArray' object has no attribute 'plev'

In [46]:
ds_plev.to_netcdf('zonal_mpsi_plev_subset.nc')
ds_hybrid.to_netcdf('zonal_mpsi_hybrid_subset.nc')

<xarray.UxDataArray 'PS' (time: 1, n_face: 30)> Size: 120B
[30 values with dtype=float32]
Coordinates:
  * time     (time) object 8B 0005-02-01 00:00:00
Dimensions without coordinates: n_face
Attributes:
    units:         Pa
    long_name:     Surface pressure
    cell_methods:  time: mean

In [51]:
ds_plev = V_sub.to_dataset()
ds_plev['PS'] = PS_sub
ds_plev

<xarray.UxDataset> Size: 4kB
Dimensions:  (lev: 32, time: 1, n_face: 30)
Coordinates:
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * time     (time) object 8B 0005-02-01 00:00:00
Dimensions without coordinates: n_face
Data variables:
    V        (time, lev, n_face) float32 4kB ...
    PS       (time, n_face) float32 120B ...

In [56]:
zonal_mpsi(ds_plev)


AttributeError: zonal_mpsi: input uxds must have either a 'plev' coordinate on V or hybrid coefficients 'hyam' and 'hybm'

In [59]:
zonal_mpsi(ds_hybrid)

ValueError: Zonal mean computations are currently only supported for face-centered data variables.

In [58]:
plev_sub = gc.interp_hybrid_to_pressure(
                ds_hybrid.V, ds_hybrid.PS, ds_hybrid.hyam, ds_hybrid.hybm, lev_dim=ds_hybrid.lev
            )

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()